# 00 — Environment Setup

This notebook sets up the NeuroForge development environment:
1. Installs all required dependencies
2. Loads and validates API keys from `.env`
3. Tests connections to all LLM providers (Groq, OpenRouter, GitHub Models)
4. Validates local tools (sentence-transformers, ChromaDB, PaddleOCR, etc.)
5. Documents free-tier rate limits

---

## 1. Install Dependencies

Run once to install all packages. After that, you can skip this cell.

In [ ]:
import subprocess
import sys

def install_requirements():
    """Install all project dependencies from requirements.txt."""
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-r", "../requirements.txt", "-q"],
        capture_output=True,
        text=True
    )
    if result.returncode == 0:
        print("All dependencies installed successfully.")
    else:
        print(f"Installation errors:\n{result.stderr}")
    return result.returncode == 0

# Uncomment the line below to install (first-time setup):
# install_requirements()

## 2. Load Environment Variables

Load API keys from `.env` file. Copy `.env.template` to `.env` and fill in your keys.

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

# Load .env from project root
env_path = Path("../.env")
if env_path.exists():
    load_dotenv(env_path)
    print(".env file loaded.")
else:
    print("WARNING: .env file not found. Copy .env.template to .env and add your API keys.")

# Validate required keys exist
REQUIRED_KEYS = ["GROQ_API_KEY", "OPENROUTER_API_KEY", "GITHUB_TOKEN"]

print("\n--- API Key Status ---")
for key in REQUIRED_KEYS:
    value = os.getenv(key)
    if value and value != f"your_{key.lower()}_here":
        print(f"  {key}: Set ({len(value)} chars)")
    else:
        print(f"  {key}: NOT SET")

## 3. Test LLM Provider Connections

Validates that each API key works by making a minimal test call.

### 3.1 Groq (Primary — Llama 4 Scout 17B)

In [ ]:
def test_groq_connection():
    """Test Groq API connection with a minimal request."""
    try:
        from groq import Groq

        api_key = os.getenv("GROQ_API_KEY")
        if not api_key or api_key == "your_groq_api_key_here":
            print("[SKIP] GROQ_API_KEY not configured.")
            return False

        client = Groq(api_key=api_key)
        response = client.chat.completions.create(
            model="meta-llama/llama-4-scout-17b-16e-instruct",
            messages=[{"role": "user", "content": "Say 'hello' in one word."}],
            max_tokens=10,
            temperature=0.0,
        )
        reply = response.choices[0].message.content.strip()
        print(f"[OK] Groq connected. Model: meta-llama/llama-4-scout-17b-16e-instruct")
        print(f"     Response: {reply}")
        print(f"     Tokens used: {response.usage.total_tokens}")
        return True

    except ImportError:
        print("[FAIL] 'groq' package not installed. Run: pip install groq")
        return False
    except Exception as e:
        print(f"[FAIL] Groq connection error: {e}")
        return False

test_groq_connection()

### 3.2 OpenRouter (Fallback — Mistral Small 3.2 24B)

In [ ]:
def test_openrouter_connection():
    """Test OpenRouter API connection with a minimal request."""
    try:
        from openai import OpenAI

        api_key = os.getenv("OPENROUTER_API_KEY")
        if not api_key or api_key == "your_openrouter_api_key_here":
            print("[SKIP] OPENROUTER_API_KEY not configured.")
            return False

        client = OpenAI(
            base_url="https://openrouter.ai/api/v1",
            api_key=api_key,
        )
        response = client.chat.completions.create(
            model="mistralai/mistral-small-3.2-24b-instruct:free",
            messages=[{"role": "user", "content": "Say 'hello' in one word."}],
            max_tokens=10,
            temperature=0.0,
        )
        reply = response.choices[0].message.content.strip()
        print(f"[OK] OpenRouter connected. Model: mistralai/mistral-small-3.2-24b-instruct:free")
        print(f"     Response: {reply}")
        return True

    except ImportError:
        print("[FAIL] 'openai' package not installed. Run: pip install openai")
        return False
    except Exception as e:
        print(f"[FAIL] OpenRouter connection error: {e}")
        return False

test_openrouter_connection()

### 3.3 GitHub Models (Lightweight — GPT-4.1-nano)

In [ ]:
def test_github_models_connection():
    """Test GitHub Models API connection with a minimal request."""
    try:
        from openai import OpenAI

        api_key = os.getenv("GITHUB_TOKEN")
        if not api_key or api_key == "your_github_token_here":
            print("[SKIP] GITHUB_TOKEN not configured.")
            return False

        client = OpenAI(
            base_url="https://models.github.ai/inference",
            api_key=api_key,
        )
        response = client.chat.completions.create(
            model="openai/gpt-4.1-nano",
            messages=[{"role": "user", "content": "Say 'hello' in one word."}],
            max_tokens=10,
            temperature=0.0,
        )
        reply = response.choices[0].message.content.strip()
        print(f"[OK] GitHub Models connected. Model: openai/gpt-4.1-nano")
        print(f"     Response: {reply}")
        return True

    except ImportError:
        print("[FAIL] 'openai' package not installed. Run: pip install openai")
        return False
    except Exception as e:
        print(f"[FAIL] GitHub Models connection error: {e}")
        return False

test_github_models_connection()

## 4. Validate Local Tools

These tools run entirely locally — no API keys required.

### 4.1 Sentence-Transformers (Embeddings)

In [ ]:
def test_sentence_transformers():
    """Validate sentence-transformers loads and produces embeddings."""
    try:
        from sentence_transformers import SentenceTransformer

        model = SentenceTransformer("all-MiniLM-L6-v2")
        embedding = model.encode("Test sentence for embedding.")
        print(f"[OK] sentence-transformers loaded.")
        print(f"     Model: all-MiniLM-L6-v2")
        print(f"     Embedding dimension: {len(embedding)}")
        return True

    except ImportError:
        print("[FAIL] 'sentence-transformers' not installed.")
        return False
    except Exception as e:
        print(f"[FAIL] sentence-transformers error: {e}")
        return False

test_sentence_transformers()

### 4.2 ChromaDB (Vector Database)

In [ ]:
def test_chromadb():
    """Validate ChromaDB can create a collection and store/query vectors."""
    try:
        import chromadb

        client = chromadb.Client()  # ephemeral in-memory client for testing
        collection = client.create_collection(name="test_collection")
        collection.add(
            documents=["This is a test document."],
            ids=["test_1"]
        )
        results = collection.query(query_texts=["test"], n_results=1)
        client.delete_collection(name="test_collection")

        print(f"[OK] ChromaDB working.")
        print(f"     Version: {chromadb.__version__}")
        print(f"     Query returned {len(results['ids'][0])} result(s).")
        return True

    except ImportError:
        print("[FAIL] 'chromadb' not installed.")
        return False
    except Exception as e:
        print(f"[FAIL] ChromaDB error: {e}")
        return False

test_chromadb()

### 4.3 PaddleOCR

In [ ]:
def test_paddleocr():
    """Validate PaddleOCR loads without errors."""
    try:
        from paddleocr import PaddleOCR

        # Initialize with show_log=False to suppress verbose output
        ocr = PaddleOCR(use_angle_cls=True, lang="en", show_log=False)
        print(f"[OK] PaddleOCR loaded.")
        print(f"     Language: English")
        print(f"     Angle classification: Enabled")
        return True

    except ImportError:
        print("[FAIL] 'paddleocr' not installed. Run: pip install paddleocr paddlepaddle")
        return False
    except Exception as e:
        print(f"[FAIL] PaddleOCR error: {e}")
        return False

test_paddleocr()

### 4.4 Document Processing Libraries

In [ ]:
def test_document_libraries():
    """Validate all document processing libraries are importable."""
    results = {}

    # pdfplumber
    try:
        import pdfplumber
        results["pdfplumber"] = f"OK (v{pdfplumber.__version__})"
    except ImportError:
        results["pdfplumber"] = "NOT INSTALLED"
    except Exception as e:
        results["pdfplumber"] = f"ERROR: {e}"

    # PyMuPDF (fitz)
    try:
        import fitz
        results["PyMuPDF (fitz)"] = f"OK (v{fitz.version[0]})"
    except ImportError:
        results["PyMuPDF (fitz)"] = "NOT INSTALLED"
    except Exception as e:
        results["PyMuPDF (fitz)"] = f"ERROR: {e}"

    # python-pptx
    try:
        import pptx
        results["python-pptx"] = f"OK (v{pptx.__version__})"
    except ImportError:
        results["python-pptx"] = "NOT INSTALLED"
    except Exception as e:
        results["python-pptx"] = f"ERROR: {e}"

    # python-docx
    try:
        import docx
        results["python-docx"] = "OK"
    except ImportError:
        results["python-docx"] = "NOT INSTALLED"
    except Exception as e:
        results["python-docx"] = f"ERROR: {e}"

    # youtube-transcript-api
    try:
        from youtube_transcript_api import YouTubeTranscriptApi
        results["youtube-transcript-api"] = "OK"
    except ImportError:
        results["youtube-transcript-api"] = "NOT INSTALLED"
    except Exception as e:
        results["youtube-transcript-api"] = f"ERROR: {e}"

    print("--- Document Processing Libraries ---")
    for lib, status in results.items():
        icon = "OK" if status.startswith("OK") else "FAIL"
        print(f"  [{icon}] {lib}: {status}")

    return all(s.startswith("OK") for s in results.values())

test_document_libraries()

### 4.5 LangChain & LangGraph

In [ ]:
def test_langchain_langgraph():
    """Validate LangChain and LangGraph are importable."""
    results = {}

    try:
        import langchain
        results["langchain"] = f"OK (v{langchain.__version__})"
    except ImportError:
        results["langchain"] = "NOT INSTALLED"
    except Exception as e:
        results["langchain"] = f"ERROR: {e}"

    try:
        import langgraph
        results["langgraph"] = f"OK (v{langgraph.__version__})"
    except ImportError:
        results["langgraph"] = "NOT INSTALLED"
    except Exception as e:
        results["langgraph"] = f"ERROR: {e}"

    try:
        import networkx
        results["networkx"] = f"OK (v{networkx.__version__})"
    except ImportError:
        results["networkx"] = "NOT INSTALLED"
    except Exception as e:
        results["networkx"] = f"ERROR: {e}"

    try:
        import pydantic
        results["pydantic"] = f"OK (v{pydantic.__version__})"
    except ImportError:
        results["pydantic"] = "NOT INSTALLED"
    except Exception as e:
        results["pydantic"] = f"ERROR: {e}"

    print("--- Framework Libraries ---")
    for lib, status in results.items():
        icon = "OK" if status.startswith("OK") else "FAIL"
        print(f"  [{icon}] {lib}: {status}")

    return all(s.startswith("OK") for s in results.values())

test_langchain_langgraph()

## 5. Full Validation Summary

In [ ]:
def run_full_validation():
    """Run all validation checks and print a summary."""
    print("=" * 60)
    print("  NEUROFORGE — ENVIRONMENT VALIDATION SUMMARY")
    print("=" * 60)
    print()

    checks = {
        "Groq API": test_groq_connection,
        "OpenRouter API": test_openrouter_connection,
        "GitHub Models API": test_github_models_connection,
        "Sentence-Transformers": test_sentence_transformers,
        "ChromaDB": test_chromadb,
        "PaddleOCR": test_paddleocr,
        "Document Libraries": test_document_libraries,
        "LangChain/LangGraph": test_langchain_langgraph,
    }

    results = {}
    for name, check_fn in checks.items():
        print(f"\n--- {name} ---")
        results[name] = check_fn()

    # Summary table
    print("\n" + "=" * 60)
    print("  RESULTS")
    print("=" * 60)
    for name, passed in results.items():
        status = "PASS" if passed else "FAIL/SKIP"
        print(f"  {status:>10}  {name}")

    passed_count = sum(results.values())
    total_count = len(results)
    print(f"\n  {passed_count}/{total_count} checks passed.")
    if passed_count == total_count:
        print("  Environment is fully configured!")
    else:
        print("  Some checks failed — see above for details.")

run_full_validation()

## 6. Free-Tier Rate Limits & Usage Notes

All services used in NeuroForge offer free tiers. Below are the limits to be aware of:

| Provider | Model | Rate Limit | Daily Limit | Notes |
|----------|-------|-----------|-------------|-------|
| **Groq** | Llama 4 Scout 17B | 30 req/min | 14,400 req/day | Primary LLM. Fast inference. |
| **OpenRouter** | Mistral Small 3.2 24B (free) | ~20 req/min | Varies | Fallback LLM. Good structured output. |
| **GitHub Models** | GPT-4.1-nano | ~15 req/min | 150 req/day | Lightweight tasks only. Strict daily cap. |
| **Sentence-Transformers** | all-MiniLM-L6-v2 | Unlimited | Unlimited | Local. No API calls. |
| **ChromaDB** | — | Unlimited | Unlimited | Local. No API calls. |
| **PaddleOCR** | — | Unlimited | Unlimited | Local. No API calls. |
| **YouTube Transcripts** | — | ~100 req/hour | Unlimited | No API key needed. IP-based throttling. |
| **LangSmith** (optional) | — | — | 5,000 traces/month | For observability/debugging. |

### Strategy

- **Primary**: Use Groq for all heavy tasks (extraction, quiz gen, chat)
- **Fallback**: If Groq rate-limits, overflow to OpenRouter
- **Lightweight**: Use GitHub Models only for classification/routing (saves quota)
- **Local**: Embeddings, vector DB, OCR, and graph are all local — no limits
- **Backoff**: Implement exponential backoff + provider fallback on 429 errors

### Tips to Stay Within Free Tiers

1. Cache LLM responses during development (avoid re-calling for same input)
2. Use `max_tokens` conservatively — don't request 4096 if 512 suffices
3. Batch related extractions into single prompts where possible
4. Use GitHub Models (GPT-4.1-nano) for binary/classification tasks
5. Run embedding-heavy operations locally without worrying about limits

## 7. Quick Reference — Model Names

Use these exact model identifiers in API calls:

```python
MODELS = {
    "groq": "meta-llama/llama-4-scout-17b-16e-instruct",
    "openrouter": "mistralai/mistral-small-3.2-24b-instruct:free",
    "github": "openai/gpt-4.1-nano",
}

ENDPOINTS = {
    "groq": "https://api.groq.com/openai/v1",
    "openrouter": "https://openrouter.ai/api/v1",
    "github": "https://models.github.ai/inference",
}
```

---

**Setup complete.** Proceed to `01_document_ingestion.ipynb` to start building the ingestion pipeline.